# SDM behind a 90% confidence gate

[Notebook 05](05_sdm_resnet18_cal100.ipynb) fitted SDM on all 100 classes and it formed **no region**: at α = 0.90, the best cutoff still left 17 classes failing. Those failing classes contain many low-confidence images the ResNet gets wrong.

This notebook puts a **selection pool** in front of SDM, as a two-stage rule:

1. **Gate.** For each **predicted** class, a temperature-scaled softmax confidence threshold chosen so that the images it admits are at least **90% correct**. The gate uses the prediction, not the true label, so it can be applied to new images. A predicted class that never reaches 90% is always rejected.
2. **SDM.** Calibrated and evaluated **only on images that pass the gate**.

`SPLIT_CALIBRATION` chooses which calibration images each stage uses:

- `True`: the 100 calibration images per class are split 50/50. The **gate half** chooses the per-class thresholds, and the **SDM half**, after gating, calibrates SDM. The α estimates stay honest, but SDM gets only about 30 gated images per class, and in our run it formed no region.
- `False` (default): both stages use **all 100** calibration images per class, so SDM gets about twice as many gated images. Because the gate is chosen on the same images SDM calibrates on, SDM's α estimates are **optimistic**; treat this run as exploratory.

The validation split is only used for the final numbers, after passing through the same gate. Any new image must pass the gate first, then SDM.

**Before running:** run notebook 04 (checkpoint) and preferably notebook 05 (embeddings cache, and the ungated SDM model used as a reference). The gated SDM fit takes about 5–10 minutes the first time and is cached in `outputs/`.

**Evaluation audit — October 29, 2026:** thresholds and risk curves now use the shared whole-confidence-group metrics. Saved outputs are cleared because historical tables used older definitions. Calibration-matched comparisons freeze the coverage budget on calibration only; validation coverages may differ. This source repair does not rerun or validate historical benchmark claims.


## 1. Settings and tools

In [ ]:
from pathlib import Path
import sys

REPO_ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / "src/sdm_vision").is_dir())
sys.path.insert(0, str(REPO_ROOT / "src"))
from sdm_vision.metrics import choose_threshold
from sdm_vision.sdm import validate_calibration_identity
import hashlib
import pickle
import tarfile
import time

import kagglehub
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import reexpress_sdm
import torch
import torch.nn as nn
import torch.nn.functional as F
from reexpress_sdm import SDMModel, TorchTrainer, TrainingConfig, load_artifact, write_artifact

SDM_EPOCHS = 20        # the package default
SDM_SEED = 0
TARGET_ALPHA = 0.95    # the reliability level we want, as in notebook 02
MODEL_NAME = "cifar100_resnet18_cal100"  # saved by notebook 04; notebook 03 uses "cifar100_resnet18" (25 calibration images per class)
SAVE_SDM_MODELS = True    # cache fitted SDM models in outputs/ (about 190 MB each) and reload them instead of refitting (about 5 minutes each)
REBUILD_EMBEDDINGS = False  # True recomputes the embeddings even when a matching cache exists
OUTPUT_DIR = Path("../outputs")
GATE_TARGET = 0.90   # accuracy each predicted class must reach to pass the gate
GATE_SEED = 0        # seed for the 50/50 split of the calibration images
SPLIT_CALIBRATION = False  # True: gate and SDM use separate halves; False: both use all calibration images

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
print(f"PyTorch {torch.__version__}, reexpress-sdm {reexpress_sdm.__version__}, device: {device}")

## 2. Load the data and the trained ResNet-18

The data loader is the same as in the earlier notebooks. The network class must match notebook 04 exactly so the saved weights fit. The checkpoint also stores the split indices, so we use exactly the same training, calibration, and validation images.

In [ ]:
DATA_DIR = Path("../data")  # the repository's data/ folder
data_path = DATA_DIR / "cifar-100-python"

if not data_path.exists():
    archive_path = DATA_DIR / "cifar-100-python.tar.gz"
    if archive_path.exists():
        with tarfile.open(archive_path) as archive:
            archive.extractall(DATA_DIR, filter="data")
    else:
        data_path = Path(kagglehub.dataset_download("aakash119/cifar-100-python-tar-gz")) / "cifar-100-python"

expected_md5 = {
    "train": "16019d7e3df5f24257cddd939b257f8d",
    "meta": "7973b15100ade9c7d40fb424638fde48",
}
for name, md5 in expected_md5.items():
    assert hashlib.md5((data_path / name).read_bytes()).hexdigest() == md5, f"{name} is not the official file; delete {data_path} and rerun"

with (data_path / "train").open("rb") as file:
    training = pickle.load(file, encoding="latin1")
with (data_path / "meta").open("rb") as file:
    metadata = pickle.load(file, encoding="latin1")
all_images = training["data"].reshape(-1, 3, 32, 32)
all_labels = np.array(training["fine_labels"])
all_groups = np.array(training["coarse_labels"])
class_names = metadata["fine_label_names"]
group_names = metadata["coarse_label_names"]

In [ ]:
# Same architecture as notebook 04.
class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, 3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, 3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.shortcut = nn.Identity()
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, 1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels),
            )

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        return F.relu(out + self.shortcut(x))


class CifarResNet(nn.Module):
    def __init__(self, num_classes=100, widths=(64, 128, 256, 512)):
        super().__init__()
        self.stem = nn.Sequential(nn.Conv2d(3, widths[0], 3, padding=1, bias=False), nn.BatchNorm2d(widths[0]), nn.ReLU())
        stages, in_channels = [], widths[0]
        for i, width in enumerate(widths):
            stride = 1 if i == 0 else 2
            stages.append(nn.Sequential(ResidualBlock(in_channels, width, stride), ResidualBlock(width, width)))
            in_channels = width
        self.stages = nn.Sequential(*stages)
        self.classifier = nn.Linear(widths[-1], num_classes)

    def embed(self, x):
        return F.adaptive_avg_pool2d(self.stages(self.stem(x)), 1).flatten(1)

    def forward(self, x):
        return self.classifier(self.embed(x))


checkpoint_path = OUTPUT_DIR / f"{MODEL_NAME}.pt"
checkpoint_sha256 = hashlib.sha256(checkpoint_path.read_bytes()).hexdigest()  # identifies this exact ResNet for the caches
checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
model = CifarResNet(num_classes=len(class_names)).to(device)
model.load_state_dict(checkpoint["model_state"])
model.eval()
channel_mean = checkpoint["channel_mean"].to(device)
channel_std = checkpoint["channel_std"].to(device)
splits = checkpoint["split_indices"]          # train / validation / calibration indices into the training file
TEMPERATURE = checkpoint["temperature"]       # fitted in notebook 04 on the calibration split
print({name: len(index) for name, index in splits.items()}, f"temperature {TEMPERATURE:.2f}")

## 3. Compute embeddings

SDM works on the **512-number embedding** each image gets just before the ResNet's final layer. We compute it, plus the logits, for every image in each split. No augmentation is applied.

The result is cached in `outputs/<MODEL_NAME>_embeddings.npz` together with a fingerprint (SHA-256) of the checkpoint. A later run loads the cache instead of recomputing, but only if the fingerprint and the split indices match, so a retrained ResNet never reuses stale embeddings. Set `REBUILD_EMBEDDINGS = True` to force a recompute.

The **training** embeddings become SDM's *support set*: the reference images it searches for nearest neighbors. The **calibration** embeddings fit SDM's regions. The **validation** embeddings are only used to check the result.

In [ ]:
@torch.no_grad()
def embed_images(images, batch_size=500):
    embeddings, logits = [], []
    for start in range(0, len(images), batch_size):
        batch = torch.from_numpy(images[start : start + batch_size]).to(device)
        features = model.embed((batch.float() / 255 - channel_mean) / channel_std)
        embeddings.append(features.cpu())
        logits.append(model.classifier(features).cpu())
    return torch.cat(embeddings).numpy(), torch.cat(logits)


embeddings_path = OUTPUT_DIR / f"{MODEL_NAME}_embeddings.npz"


def load_cached_embeddings():
    # The cache is valid only for this exact checkpoint and these exact splits.
    if REBUILD_EMBEDDINGS or not embeddings_path.exists():
        return None
    with np.load(embeddings_path) as cache:
        if "checkpoint_sha256" not in cache.files or str(cache["checkpoint_sha256"]) != checkpoint_sha256:
            return None
        if any(not np.array_equal(cache[f"{name}_index"], index) for name, index in splits.items()):
            return None
        return {name: {"index": cache[f"{name}_index"], "embeddings": cache[f"{name}_embeddings"],
                       "logits": torch.from_numpy(cache[f"{name}_logits"]),
                       "labels": cache[f"{name}_labels"], "groups": cache[f"{name}_groups"]}
                for name in splits}


data = load_cached_embeddings()
if data is None:
    data = {}
    for name, index in splits.items():
        embeddings, logits = embed_images(all_images[index])
        data[name] = {"index": index, "embeddings": embeddings, "logits": logits,
                      "labels": all_labels[index], "groups": all_groups[index]}
    np.savez_compressed(
        embeddings_path, checkpoint_sha256=checkpoint_sha256,
        **{f"{name}_{key}": (value.numpy() if torch.is_tensor(value) else value)
           for name, split in data.items() for key, value in split.items()},
    )
    print("Computed embeddings and cached them in", embeddings_path)
else:
    print("Loaded cached embeddings from", embeddings_path)

pd.DataFrame({
    name: {"images": len(split["labels"]),
           "ResNet accuracy": f"{(split['logits'].argmax(1).numpy() == split['labels']).mean():.1%}"}
    for name, split in data.items()
}).T

## 4. Tools

The same tools as notebook 05, except that `fit_sdm` now takes the calibration images as an argument, so it can be calibrated on the gated images. Its support set is still the full training split with all 100 labels.

In [ ]:
def fit_sdm(number_of_classes, train_labels, calibration, calibration_labels, label_names, name):
    path = OUTPUT_DIR / f"{MODEL_NAME}_{name}_epochs{SDM_EPOCHS}_seed{SDM_SEED}.sdmkitmodel"
    fingerprint = f"{MODEL_NAME}_avgpool512_{checkpoint_sha256[:16]}"
    start = time.time()
    artifact = load_artifact(path) if SAVE_SDM_MODELS and path.exists() else None
    if artifact is not None and artifact.manifest["representation"]["fingerprint"] == fingerprint:
        validate_calibration_identity(artifact, [f"train-{i}" for i in calibration["index"]], calibration_labels, label_names)
        source = f"loaded from {path.name}"
    else:
        trainer = TorchTrainer(TrainingConfig(number_of_classes=number_of_classes, epochs=SDM_EPOCHS, seed=SDM_SEED), device=str(device))
        result = trainer.fit(
            data["train"]["embeddings"], train_labels.tolist(),
            calibration["embeddings"], calibration_labels.tolist(),
            train_ids=[f"train-{i}" for i in data["train"]["index"]],
            calibration_ids=[f"train-{i}" for i in calibration["index"]],
            class_names=label_names,
            representation_fingerprint=fingerprint,
        )
        artifact = result.artifact
        source = f"best epoch {result.best_epoch}"
        if SAVE_SDM_MODELS:
            write_artifact(path, artifact, overwrite=path.exists())
    regions = [round(region["alpha"], 2) for region in artifact.manifest["regions"]]
    print(f"{name}: {source}, {time.time() - start:.0f}s, regions at alpha = {regions or 'none'}")
    return artifact


def region_ladder(artifact, number_of_classes):
    # Replays SDM's nested calibration. At each alpha of the ladder, find the similarity cutoff that leaves
    # the fewest failing labels among the calibration images no earlier region took; if SDM formed a region
    # at that alpha, remove the images it accepts before moving on, exactly as the package does.
    rows = artifact.calibration_rows
    probs = np.array([row["sdm"] for row in rows], dtype=np.float32)
    q_prime = np.array([row["qPrime"] for row in rows], dtype=np.float32)
    labels = np.array([int(row["label"]) for row in rows])
    predictions = np.array([int(row["prediction"]) for row in rows])
    p_true = probs[np.arange(len(labels)), labels]
    configuration = artifact.manifest["configuration"]
    regions = {round(region["alpha"], 10): region for region in artifact.manifest["regions"]}
    residual = np.floor(q_prime) > configuration["oodLimit"]
    ladder = {}
    for step in range(1, round(0.5 / configuration["alphaResolution"]) + 1):
        alpha = round(1 - step * configuration["alphaResolution"], 10)
        if alpha <= 0.5 or not residual.any():
            break
        below = p_true < np.float32(alpha)
        best = None
        for cutoff in np.unique(q_prime[residual]):
            kept = residual & (q_prime >= cutoff)
            count = np.bincount(labels[kept], minlength=number_of_classes)
            misses = np.bincount(labels[kept & below], minlength=number_of_classes)
            allowed = np.floor(round(1 - alpha, 10) * count).astype(int)
            failing = (count == 0) | (misses > allowed)
            if best is None or failing.sum() < best["failing"].sum():
                best = {"failing": failing, "count": count, "misses": misses, "allowed": allowed}
        region = regions.get(alpha)
        ladder[alpha] = best | {"region": region, "rows left": int(residual.sum())}
        if region is not None:
            above = probs >= np.asarray(region["outputThresholds"], dtype=np.float32)
            accepted = (residual & (q_prime >= np.float32(region["minimumRescaledSimilarity"]))
                        & (above.sum(1) == 1) & above[np.arange(len(labels)), predictions])
            residual &= ~accepted
    return ladder


def region_blockers(artifact, number_of_classes):
    # For each alpha, the fewest failing labels at the best cutoff; 0 failing means SDM formed a region there.
    if not any(np.floor(row["qPrime"]) > artifact.manifest["configuration"]["oodLimit"] for row in artifact.calibration_rows):
        return pd.DataFrame([{"region formed": "", "calibration images left": 0,
                              "labels with no images left": number_of_classes, "labels with too many misses": 0}],
                            index=pd.Index(["none eligible"], name="alpha"))
    table = {}
    for alpha, step in region_ladder(artifact, number_of_classes).items():
        table[f"{alpha:.2f}"] = {
            "region formed": "yes" if step["region"] else "",
            "calibration images left": step["rows left"],
            "labels with no images left": int((step["count"] == 0).sum()),
            "labels with too many misses": int(((step["count"] > 0) & (step["misses"] > step["allowed"])).sum()),
        }
    return pd.DataFrame(table).T.rename_axis("alpha")


def wilson_interval(successes, n, z=1.96):
    p = successes / n
    denominator = 1 + z**2 / n
    center = (p + z**2 / (2 * n)) / denominator
    half_width = z * np.sqrt(p * (1 - p) / n + z**2 / (4 * n**2)) / denominator
    return center - half_width, center + half_width


def selective_summary(admitted, predictions, labels):
    n = int(admitted.sum())
    correct = predictions[admitted] == labels[admitted]
    if n == 0:
        return {"coverage": "0.0%", "images answered": 0, "selective accuracy": "-", "95% interval": "-", "worst label accuracy": "-"}
    low, high = wilson_interval(correct.sum(), n)
    per_label = pd.Series(correct).groupby(labels[admitted]).mean()
    return {
        "coverage": f"{admitted.mean():.1%}",
        "images answered": n,
        "selective accuracy": f"{correct.mean():.1%}",
        "95% interval": f"{low:.1%} – {high:.1%}",
        "worst label accuracy": f"{per_label.min():.0%} ({len(per_label)} labels)",
    }




def score_validation(sdm, nearest_exemplars=0):
    # `sdm` is an SDMModel kept in memory, so its nearest-neighbor index is built only once.
    scores = sdm.score(data["validation"]["embeddings"], ids=[f"train-{i}" for i in data["validation"]["index"]],
                       nearest_exemplars=nearest_exemplars)
    return {
        "scores": scores,
        "prediction": np.array([score.prediction for score in scores]),
        "centroid_alpha": np.array([score.centroid_region_alpha for score in scores]),
        "lower_alpha": np.array([score.lower_region_alpha for score in scores]),
        "is_ood": np.array([score.is_ood for score in scores]),
    }

## 5. Calibration images for the gate and for SDM

With `SPLIT_CALIBRATION = True`, a split stratified by true class: 50 images per class go to the **gate half** and 50 to the **SDM half**. Otherwise both stages use all calibration images.

In [ ]:
def subset(split, mask):
    # Logits are a torch tensor, everything else a NumPy array.
    return {key: value[torch.from_numpy(mask)] if torch.is_tensor(value) else value[mask] for key, value in split.items()}


if SPLIT_CALIBRATION:
    rng = np.random.default_rng(GATE_SEED)
    in_gate_half = np.zeros(len(data["calibration"]["labels"]), dtype=bool)
    for k in range(100):
        members = np.flatnonzero(data["calibration"]["labels"] == k)
        in_gate_half[rng.choice(members, size=len(members) // 2, replace=False)] = True
    data["gate"] = subset(data["calibration"], in_gate_half)
    data["sdm_cal"] = subset(data["calibration"], ~in_gate_half)
    assert not set(data["gate"]["index"]) & set(data["sdm_cal"]["index"])
    assert len(data["gate"]["index"]) + len(data["sdm_cal"]["index"]) == len(data["calibration"]["index"])
else:
    data["gate"] = data["sdm_cal"] = data["calibration"]
print({name: len(data[name]["labels"]) for name in ["gate", "sdm_cal"]},
      "images per class:", np.bincount(data["gate"]["labels"]).min(), "/", np.bincount(data["sdm_cal"]["labels"]).min())

## 6. The 90% gate, one threshold per predicted class

On the gate images, for each predicted class we sort its images by confidence and keep the lowest threshold at which the admitted images are still at least 90% correct (`choose_threshold`, as in notebook 02). If no threshold reaches 90%, the class gets an infinite threshold and the gate always rejects it.

In [ ]:
def softmax_confidence(logits):
    probs = (logits / TEMPERATURE).softmax(1).numpy()
    return probs.max(1), probs.argmax(1)


gate_conf, gate_pred = softmax_confidence(data["gate"]["logits"])
gate_correct = gate_pred == data["gate"]["labels"]
gate_thresholds = np.full(100, np.inf)
for k in range(100):
    predicted_k = gate_pred == k
    if predicted_k.any():
        threshold = choose_threshold(gate_conf[predicted_k], gate_correct[predicted_k], GATE_TARGET)
        if threshold is not None:
            gate_thresholds[k] = threshold


def gate_pass(logits):
    confidence, prediction = softmax_confidence(logits)
    return confidence >= gate_thresholds[prediction]


passed = gate_pass(data["gate"]["logits"])
per_class = pd.Series(gate_correct[passed]).groupby(gate_pred[passed]).mean()
assert (per_class >= GATE_TARGET).all()  # by construction, on the gate images

always_rejected = [class_names[k] for k in np.flatnonzero(np.isinf(gate_thresholds))]
print(f"gate images: {passed.mean():.1%} of images pass, accuracy among them {gate_correct[passed].mean():.1%}")
print(f"predicted classes always rejected ({len(always_rejected)}): {', '.join(always_rejected) or 'none'}")
pd.Series(gate_thresholds, index=class_names).replace(np.inf, np.nan).describe().rename("threshold").to_frame().T

## 7. The gated SDM calibration set

SDM's calibration images go through the gate. Only the images that pass are used to calibrate SDM. Classes with fewer than 20 images left tolerate **no misses** at α = 0.95 and few at 0.90, so they are the likely blockers.

In [ ]:
sdm_cal_pass = gate_pass(data["sdm_cal"]["logits"])
data["sdm_cal_gated"] = subset(data["sdm_cal"], sdm_cal_pass)

kept = np.bincount(data["sdm_cal_gated"]["labels"], minlength=100)
_, sdm_cal_pred = softmax_confidence(data["sdm_cal_gated"]["logits"])
print(f"SDM calibration images: {sdm_cal_pass.mean():.1%} pass the gate ({sdm_cal_pass.sum()} images), "
      f"ResNet accuracy among them {(sdm_cal_pred == data['sdm_cal_gated']['labels']).mean():.1%}")
print(f"images per true class after the gate: median {np.median(kept):.0f}, min {kept.min()}, "
      f"classes with fewer than 20: {(kept < 20).sum()}, with none: {(kept == 0).sum()}")

## 8. Fit SDM on the gated calibration set

The labels and support set are unchanged: 100 fine classes, all training images. Only the calibration images differ from notebook 05.

In [ ]:
sdm_gated = fit_sdm(100, data["train"]["labels"], data["sdm_cal_gated"], data["sdm_cal_gated"]["labels"],
                    class_names, f"sdm_fine100_gated{round(GATE_TARGET * 100)}_" + (f"split{GATE_SEED}" if SPLIT_CALIBRATION else "fullcal"))
gated_scorer = SDMModel(sdm_gated, device=str(device))
gated = score_validation(gated_scorer)
y_val = data["validation"]["labels"]
val_gate = gate_pass(data["validation"]["logits"])
print(f"validation images passing the gate: {val_gate.mean():.1%}")
print(f"validation images in any SDM region (alpha > 0): {(gated['centroid_alpha'] > 0).mean():.1%}")

If SDM still formed no region, the table below shows how close each α came: the fewest failing classes at the best cutoff. Compare it with notebook 05 (at α = 0.90: 10 classes with no images left, 7 with too many misses).

In [ ]:
region_blockers(sdm_gated, 100)

## 9. Results on validation (100 fine classes)

Every row is a rule you could apply to a new image. The global softmax threshold is chosen on the gate images to reach 90% overall, as in notebook 02. The ungated SDM row is notebook 05's model, loaded from its cache.

In [ ]:
val_conf, val_pred = softmax_confidence(data["validation"]["logits"])
global_threshold = choose_threshold(gate_conf, gate_correct, GATE_TARGET)

rows = {
    "softmax, one global threshold": selective_summary(val_conf >= global_threshold, val_pred, y_val),
    "gate alone (per-class thresholds)": selective_summary(val_gate, val_pred, y_val),
}
# The gate's threshold belongs to the ResNet's predicted class, so only answer when SDM predicts the same class.
agree = gated["prediction"] == val_pred
print(f"SDM and the ResNet predict the same class on {agree.mean():.1%} of validation images")
for alpha in [0.90, 0.95]:
    rows[f"gate + SDM, centroid alpha ≥ {alpha:.2f}"] = selective_summary(val_gate & agree & (gated["centroid_alpha"] >= alpha), gated["prediction"], y_val)
    rows[f"gate + SDM, lower alpha ≥ {alpha:.2f}"] = selective_summary(val_gate & agree & (gated["lower_alpha"] >= alpha), gated["prediction"], y_val)

ungated_path = OUTPUT_DIR / f"{MODEL_NAME}_sdm_fine100_epochs{SDM_EPOCHS}_seed{SDM_SEED}.sdmkitmodel"
ungated_artifact = load_artifact(ungated_path) if ungated_path.exists() else None
if ungated_artifact is not None and ungated_artifact.manifest["representation"]["fingerprint"] == f"{MODEL_NAME}_avgpool512_{checkpoint_sha256[:16]}":
    ungated = score_validation(SDMModel(ungated_artifact, device=str(device)))
    rows["SDM without gate (notebook 05), centroid alpha ≥ 0.90"] = selective_summary(ungated["centroid_alpha"] >= 0.90, ungated["prediction"], y_val)

pd.DataFrame(rows).T.rename_axis("validation, 100 classes")

**How to read this table:**

- **Gate alone** is the selection pool itself. On new images, expect overall accuracy near 90% but not every class at 90%: its thresholds were chosen on only 50 images per class, and validation has only 25 per class, so the worst-label column is noisy.
- **Gate + SDM** answers only images that pass both stages and on which SDM predicts the same class as the ResNet (the gate's threshold belongs to the ResNet's predicted class; SDM's own prediction differs on roughly one image in six). SDM's α is a class-conditional empirical criterion *within the gated pool*, so it only holds for images that went through the same gate.
- If SDM formed no region, the gate + SDM rows answer nothing. Then `region_blockers` above shows what is left to fix: usually classes with too few gated calibration images. Two follow-ups: apply the same gate on the 20 groups (500 calibration images each), or merge the confusable classes (for example boy/girl/man/woman).